In [ ]:

import requests

BASE_URL = "http://127.0.0.1:8000"

print("🔐 AI Agent 방탈출 게임")
print("=" * 45)
print("비밀 금고를 열려면 세 가지 암호를 찾아야 합니다.")
print("Qwen3-8B Agent가 Tool을 선택해 암호를 계산합니다.")

# 세 가지 미션
missions = [
    {
        "title": "첫 번째 암호 - 계산",
        "question": "37 곱하기 4에 12를 더해줘.",
        "tool": "calculator",
        "key": "result"
    },
    {
        "title": "두 번째 암호 - 글자 수",
        "question": "'AI AGENT'의 글자 수를 계산해줘. 공백도 포함해.",
        "tool": "text_length",
        "key": "result"
    },
    {
        "title": "세 번째 암호 - 데이터 분석",
        "question": "[5,15,25]의 합계와 평균을 알려줘.",
        "tool": "data_summary",
        "key": "sum"
    }
]

# FastAPI 서버 확인
health = requests.get(BASE_URL + "/health", timeout=10)
health.raise_for_status()

if not health.json().get("cuda_available"):
    raise RuntimeError("GPU가 연결되지 않았습니다.")

print("✅ FastAPI / GPU 확인 완료")

password_parts = []

for i, mission in enumerate(missions, start=1):

    print(f"\n🎯 미션 {i}: {mission['title']}")
    print("질문:", mission["question"])
    print("🤖 AI Agent가 암호를 찾는 중...")

    try:
        response = requests.post(
            BASE_URL + "/agent",
            json={"query": mission["question"]},
            timeout=240
        )
        response.raise_for_status()
        data = response.json()

    except requests.RequestException as e:
        print("❌ 서버 요청 실패:", e)
        break

    tool_result = data.get("tool_result")

    # LLM 판단과 Tool 실행 성공 여부 검사
    if (
        data.get("status") != "success"
        or data.get("tool_name") != mission["tool"]
        or not isinstance(tool_result, dict)
        or not tool_result.get("ok")
    ):
        print("❌ 미션 실패!")
        print("AI 판단:", data.get("decision"))
        print("실행 상태:", data.get("status"))
        print("실제 Tool:", data.get("tool_name"))
        break

    # Tool의 실제 실행 결과에서 암호 추출
    result = tool_result["result"]

    if mission["tool"] == "data_summary":
        secret = result["sum"]
    else:
        secret = result

    password_parts.append(str(int(secret)))

    print("✅ 미션 성공!")
    print("사용 Tool:", data["tool_name"])
    print("Tool 결과:", result)
    print("획득한 암호:", secret)

# 세 가지 미션에 모두 성공한 경우
if len(password_parts) == len(missions):

    password = "-".join(password_parts)

    print("\n🔓 세 개의 암호를 모두 획득했습니다!")
    print("암호 입력 형식: 숫자-숫자-숫자")

    guess = input("🔐 비밀 금고 암호를 입력하세요: ").strip()

    if guess == password:
        print("\n🎉 금고 열기 성공!")
        print("🏆 AI Agent 방탈출 미션 완료!")
    else:
        print("\n❌ 암호가 일치하지 않습니다.")
        print("정답 암호:", password)

else:
    print("\n⚠️ 미션이 중단되었습니다.")
    print("오류 원인을 확인한 후 다시 도전하세요.")


🔐 AI Agent 방탈출 게임
비밀 금고를 열려면 세 가지 암호를 찾아야 합니다.
Qwen3-8B Agent가 Tool을 선택해 암호를 계산합니다.
✅ FastAPI / GPU 확인 완료

🎯 미션 1: 첫 번째 암호 - 계산
질문: 37 곱하기 4에 12를 더해줘.
🤖 AI Agent가 암호를 찾는 중...
✅ 미션 성공!
사용 Tool: calculator
Tool 결과: 160
획득한 암호: 160

🎯 미션 2: 두 번째 암호 - 글자 수
질문: 'AI AGENT'의 글자 수를 계산해줘. 공백도 포함해.
🤖 AI Agent가 암호를 찾는 중...
✅ 미션 성공!
사용 Tool: text_length
Tool 결과: 8
획득한 암호: 8

🎯 미션 3: 세 번째 암호 - 데이터 분석
질문: [5,15,25]의 합계와 평균을 알려줘.
🤖 AI Agent가 암호를 찾는 중...
✅ 미션 성공!
사용 Tool: data_summary
Tool 결과: {'count': 3, 'sum': 45, 'average': 15.0, 'min': 5, 'max': 25}
획득한 암호: 45

🔓 세 개의 암호를 모두 획득했습니다!
암호 입력 형식: 숫자-숫자-숫자
